In [ ]:
import yfinance as yf
import pandas as pd
import requests
import time
from datetime import datetime, timedelta

# ================= 配置区 =================
# 填入你的 Discord Webhook
import os
WEBHOOK_URL = os.environ.get("DISCORD_WEBHOOK_URL", "")
TOTAL_CAPITAL = 1000.0         # 模拟小本金 $1000
RISK_TOLERANCE = 0.02          # 小本金稍微激进一点：单次容忍 2% 亏损 ($20)

# 用于记录上一次发送通知的时间，防止每10分钟重复轰炸
alert_cooldowns = {}
COOLDOWN_HOURS = 2             # 触发通知后，同一个标的静默 2 小时
# ==========================================

def send_discord_msg(msg):
    data = {"content": msg}
    requests.post(WEBHOOK_URL, json=data)

def check_intraday_signals(ticker):
    current_time_str = datetime.now().strftime('%H:%M:%S')

    # 1. 精确打击：获取过去 5 天的 15 分钟级别数据
    stock = yf.Ticker(ticker)
    df = stock.history(period="5d", interval="15m")

    if df.empty or len(df) < 200:
        print(f"[{current_time_str}] {ticker} 盘前或无足够数据，静默。")
        return

    # 2. 计算 15 分钟级别的高敏均线
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    latest = df.iloc[-1]
    close_price = latest['Close']

    # 判断防骚扰冷却时间
    if ticker in alert_cooldowns:
        time_since_last_alert = datetime.now() - alert_cooldowns[ticker]
        if time_since_last_alert < timedelta(hours=COOLDOWN_HOURS):
            print(f"[{current_time_str}] {ticker} 现价 ${close_price:.2f} (仍在冷却期，静默中...)")
            return

    # 3. 极窄买入区间判定 (误差缩小到 0.5% 以内)
    is_above_200 = close_price > latest['EMA_200']
    is_uptrend = latest['EMA_20'] > latest['EMA_50']

    buy_zone_lower = latest['EMA_50'] * 0.998  # 极窄下限
    buy_zone_upper = latest['EMA_50'] * 1.005  # 极窄上限

    # ================= 触发逻辑 =================
    if is_above_200 and is_uptrend and (buy_zone_lower <= close_price <= buy_zone_upper):
        stop_loss_price = latest['EMA_50'] * 0.995 # 15分钟线跌破 0.5% 止损
        risk_per_share = close_price - stop_loss_price

        if risk_per_share > 0:
            suggested_shares = int((TOTAL_CAPITAL * RISK_TOLERANCE) / risk_per_share)

            # 如果是小本金，可能买不起完整的股数，建议使用支持碎股 (Fractional Shares) 的券商模型
            msg = f"🟢 **【V3.0 精确打击】** {ticker}\n" \
                  f"**现价**: ${close_price:.2f} (15分钟级别)\n" \
                  f"**趋势**: 站上 15m_EMA200 (${latest['EMA_200']:.2f})\n" \
                  f"**位置**: 极窄买入区 (${buy_zone_lower:.2f} - ${buy_zone_upper:.2f})\n" \
                  f"🛠️ **【模型建议】**\n" \
                  f"🔹 初始止损位: **${stop_loss_price:.2f}**\n" \
                  f"🔹 模拟分配股数: **{suggested_shares} 股**"

            # 只在触发时调用 Discord 发送！
            send_discord_msg(msg)
            # 记录发送时间，触发防骚扰冷却
            alert_cooldowns[ticker] = datetime.now()
            print(f"[{current_time_str}] 🚨 信号已发送至 Discord！")

    else:
        # 如果没有机会，只在本地打印日志，绝对不打扰 Discord
        print(f"[{current_time_str}] {ticker} 现价 ${close_price:.2f} | 寻找机会中，Discord 保持静默...")

if __name__ == "__main__":
    print("🤖 V3.0 小本金高频静默雷达启动...")

    # 模拟真实部署：死循环每 10 分钟跑一次
    while True:
        portfolio = ["VOO", "QQQ", "TSLA"]
        for symbol in portfolio:
            try:
                check_intraday_signals(symbol)
            except Exception as e:
                print(f"检查 {symbol} 时发生错误: {e}")

            time.sleep(2) # 标的之间暂停2秒防限流

        print("⏳ 这一轮检查完毕，休眠 10 分钟 (600秒)...\n")
        time.sleep(600) # 休眠 600 秒 (10分钟) 后自动重启循环

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_15m_backtest(ticker="QQQ", period="60d"):
    print(f"🚀 开始 V3.0 小本金回测: {ticker} (最近 {period}, 15分钟级别)")

    # 1. 获取 15 分钟级别高频数据 (Yahoo Finance 最多支持过去 60 天)
    df = yf.download(ticker, period=period, interval="15m")
    if df.empty or len(df) < 200:
        print("数据下载失败或数据不足！")
        return

    # 2. 计算量化指标 (V3.0 逻辑)
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # 3. 极窄区间买入逻辑 (买入条件: 价格大于EMA200, 且EMA20>EMA50, 且价格刚好在EMA50上方0%到0.5%之间)
    df['Buy_Signal'] = np.where(
        (df['Close'] > df['EMA_200']) &
        (df['EMA_20'] > df['EMA_50']) &
        (df['Close'] <= df['EMA_50'] * 1.005) &
        (df['Close'] >= df['EMA_50'] * 0.998),
        1, 0
    )

    # 严格防守卖出逻辑 (跌破 EMA50 的 0.5% 即止损)
    df['Sell_Signal'] = np.where((df['Close'] < df['EMA_50'] * 0.995), -1, 0)

    # 💡 修正点：使用 pandas 安全的 .add() 方法合并信号，避免对齐报错
    df['Signal'] = df['Buy_Signal'].add(df['Sell_Signal'], fill_value=0)

    # 4. 模拟持仓状态 (1 为持仓，0 为空仓)
    df['Position'] = df['Signal'].replace(to_replace=0, method='ffill')
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 5. 计算收益
    df['Market_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Position'].shift(1) * df['Market_Return']

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    # ================= 输出回测报告 =================
    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    drawdown = (df['Strategy_Equity'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100

    print("\n" + "="*40)
    print("📉 V3.0 高频回测结果 (Educational Purpose)")
    print("="*40)
    print(f"标的: {ticker} (15m Interval)")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*40)

if __name__ == "__main__":
    # 你可以在这里替换成你想测试的其他代码，比如 "SPY", "TSLA" 等
    run_15m_backtest("QQQ")


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_15m_backtest(ticker="QQQ", period="60d"):
    print(f"🚀 开始 V3.0 小本金回测: {ticker} (最近 {period}, 15分钟级别)")

    # 💡 核心修复点：改用 yf.Ticker(ticker).history()，彻底解决 pandas 维度对齐报错
    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval="15m")

    if df.empty or len(df) < 200:
        print("数据下载失败或数据不足！")
        return

    # 2. 计算量化指标 (V3.0 逻辑)
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # 3. 极窄区间买入逻辑
    df['Buy_Signal'] = np.where(
        (df['Close'] > df['EMA_200']) &
        (df['EMA_20'] > df['EMA_50']) &
        (df['Close'] <= df['EMA_50'] * 1.005) &
        (df['Close'] >= df['EMA_50'] * 0.998),
        1, 0
    )

    # 严格防守卖出逻辑
    df['Sell_Signal'] = np.where((df['Close'] < df['EMA_50'] * 0.995), -1, 0)

    # 合并信号
    df['Signal'] = df['Buy_Signal'] + df['Sell_Signal']

    # 4. 模拟持仓状态 (1 为持仓，0 为空仓)
    df['Position'] = df['Signal'].replace(to_replace=0, method='ffill')
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 5. 计算收益
    df['Market_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Position'].shift(1) * df['Market_Return']

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    # ================= 输出回测报告 =================
    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    drawdown = (df['Strategy_Equity'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100

    print("\n" + "="*40)
    print("📉 V3.0 高频回测结果 (Educational Purpose)")
    print("="*40)
    print(f"标的: {ticker} (15m Interval)")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*40)

if __name__ == "__main__":
    run_15m_backtest("QQQ")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_15m_backtest_v4(ticker="QQQ", period="60d"):
    print(f"🚀 开始 V4.0 趋势追踪回测: {ticker} (最近 {period}, 15分钟级别)")

    # 获取数据
    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval="15m")

    if df.empty or len(df) < 200:
        print("数据下载失败或数据不足！")
        return

    # 计算指标
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # 🟢 V4.0 放宽买入逻辑：只要大趋势向上，且价格在 EMA20 和 EMA50 构成的“上升通道”内，直接上车！
    # 允许最高买入价偏离 EMA20 上方 0.5%，不错过强势起飞
    df['Buy_Signal'] = np.where(
        (df['Close'] > df['EMA_200']) &
        (df['EMA_20'] > df['EMA_50']) &
        (df['Close'] <= df['EMA_20'] * 1.005) &  # 提高了买入上限
        (df['Close'] >= df['EMA_50'] * 0.995),   # 稍微放宽了下限
        1, 0
    )

    # 🔴 V4.0 放宽防守逻辑：有效跌破 EMA50 超过 1.5% 才认输止损，过滤盘中假摔
    df['Sell_Signal'] = np.where((df['Close'] < df['EMA_50'] * 0.985), -1, 0)

    # 合并信号
    df['Signal'] = df['Buy_Signal'].add(df['Sell_Signal'], fill_value=0)

    # 💡 完美修复 FutureWarning：使用最新的 pandas 链式替换和填充法
    df['Position'] = df['Signal'].replace(0, np.nan).ffill().fillna(0)
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 计算收益
    df['Market_Return'] = df['Close'].pct_change()
    # T+1 收益：今天的持仓享受明天的涨跌
    df['Strategy_Return'] = df['Position'].shift(1) * df['Market_Return']

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    # ================= 输出回测报告 =================
    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    drawdown = (df['Strategy_Equity'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100

    print("\n" + "="*40)
    print("📈 V4.0 趋势追踪回测结果 (Educational Purpose)")
    print("="*40)
    print(f"标的: {ticker} (15m Interval)")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*40)

if __name__ == "__main__":
    # 跑跑 QQQ 看这次能不能追上大盘！
    run_15m_backtest_v4("QQQ")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_1h_backtest(ticker="QQQ", period="1y"):
    print(f"🚀 开始 V5.0 佛系波段回测: {ticker} (时间跨度: {period}, 1小时级别)")

    # 获取 1 小时级别数据
    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval="1h")

    if df.empty or len(df) < 200:
        print("数据下载失败或数据不足！")
        return

    # 计算 1 小时级别的均线 (时间跨度变大，均线更加平滑)
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # 🟢 V5.0 佛系买入：不去精准接针。只要长期趋势向上(>EMA200)，且短期均线金叉(EMA20>EMA50)，就持有
    df['Buy_Signal'] = np.where(
        (df['Close'] > df['EMA_200']) &
        (df['EMA_20'] > df['EMA_50']),
        1, 0
    )

    # 🔴 V5.0 佛系卖出：不管中间怎么跌，只要收盘价没有跌破 1小时级别的 EMA50，就死拿！跌破才走。
    df['Sell_Signal'] = np.where((df['Close'] < df['EMA_50']), -1, 0)

    # 合并信号
    df['Signal'] = df['Buy_Signal'].add(df['Sell_Signal'], fill_value=0)

    # 状态与收益计算
    df['Position'] = df['Signal'].replace(0, np.nan).ffill().fillna(0)
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 计算交易次数 (持仓状态发生变化)
    trades = df['Position'].diff().abs().sum() / 2

    df['Market_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Position'].shift(1) * df['Market_Return']

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    # ================= 输出回测报告 =================
    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    drawdown = (df['Strategy_Equity'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100

    print("\n" + "="*45)
    print("🧘‍♂️ V5.0 1H 佛系波段回测 (1-Year Period)")
    print("="*45)
    print(f"标的: {ticker}")
    print(f"模拟交易次数 (Trades): 约 {int(trades)} 次")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*45)

if __name__ == "__main__":
    run_1h_backtest("QQQ", period="1y")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# 纯手工编写的 RSI 算法 (面试时可以吹这个，展现底层基础)
def calculate_rsi(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def run_1h_rsi_backtest(ticker="QQQ", period="1y"):
    print(f"🚀 开始 V6.0 恐慌抄底波段回测: {ticker} (1小时级别)")

    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval="1h")

    if df.empty or len(df) < 200:
        return

    # 计算指标
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()
    df['RSI_14'] = calculate_rsi(df['Close'], period=14)

    # 🟢 恐慌抄底买入：宏观牛市 (价格>EMA200) + 散户恐慌抛售 (RSI < 30)
    df['Buy_Signal'] = np.where(
        (df['Close'] > df['EMA_200']) &
        (df['RSI_14'] < 40),
        1, 0
    )

    # 🔴 贪婪止盈卖出：散户疯狂追高 (RSI > 70)
    df['Sell_Signal'] = np.where((df['Close'] < df['EMA_50']), -1, 0)

    # 合并信号并生成持仓
    df['Signal'] = df['Buy_Signal'].add(df['Sell_Signal'], fill_value=0)
    df['Position'] = df['Signal'].replace(0, np.nan).ffill().fillna(0)
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    trades = df['Position'].diff().abs().sum() / 2

    df['Market_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Position'].shift(1) * df['Market_Return']

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    max_drawdown = ((df['Strategy_Equity'] - rolling_max) / rolling_max).min() * 100

    print("\n" + "="*45)
    print("🎯 V6.0 RSI 均值回归波段 (Mean Reversion)")
    print("="*45)
    print(f"标的: {ticker}")
    print(f"模拟交易次数: 约 {int(trades)} 次")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*45)

if __name__ == "__main__":
    run_1h_rsi_backtest("QQQ", period="1y")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_v7_diamond_hands_engine():
    # 同时获取 QQQ 和 TQQQ 的数据
    qqq = yf.Ticker("QQQ").history(period="2y", interval="1h")
    tqqq = yf.Ticker("TQQQ").history(period="2y", interval="1h")

    # 获取 QQQ 日线数据用于“钻石手”防守
    qqq_daily = yf.Ticker("QQQ").history(period="2y", interval="1d")

    # 1. 计算 1小时级别 EMA30 (作为精准买点)
    qqq['EMA_30'] = qqq['Close'].ewm(span=30, adjust=False).mean()
    tqqq['EMA_30'] = tqqq['Close'].ewm(span=30, adjust=False).mean()

    # 2. 计算日线级别 EMA50 (作为底线防守)
    qqq_daily['Daily_EMA_50'] = qqq_daily['Close'].ewm(span=50, adjust=False).mean()

    # ================= 核心信号生成 =================

    # 信号 A：常规建仓 (QQQ 回踩 EMA30)
    qqq_hits_ema30 = (qqq['Close'] >= qqq['EMA_30'] * 0.995) & (qqq['Close'] <= qqq['EMA_30'] * 1.005)

    # 信号 B：共振狠狠加仓 (QQQ 和 TQQQ 同步回踩 EMA30)
    tqqq_hits_ema30 = (tqqq['Close'] >= tqqq['EMA_30'] * 0.995) & (tqqq['Close'] <= tqqq['EMA_30'] * 1.005)
    aggressive_buy_signal = qqq_hits_ema30 & tqqq_hits_ema30

    # 信号 C：钻石手卖出 (只有当日线级别的收盘价彻底跌破日线 EMA50 才卖出)
    # 这极大地解决了“散户拿不住被骗”的问题
    # ... (合并日线与小时线信号的底层逻辑) ...

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def run_v7_diamond_hands_backtest(period="730d"):
    print("🚀 开始 V7.0 钻石手 & 跨资产共振回测 (1h 级别结合日线防守)")
    print("📥 正在拉取 QQQ 和 TQQQ 过去约 2 年的数据...")

    # 1. 获取 1 小时级别数据 (用于寻找精准买点)
    qqq_1h = yf.Ticker("QQQ").history(period=period, interval="1h")
    tqqq_1h = yf.Ticker("TQQQ").history(period=period, interval="1h")

    # 获取 QQQ 日线数据 (用于底层“钻石手”防守)
    qqq_1d = yf.Ticker("QQQ").history(period=period, interval="1d")

    if qqq_1h.empty or tqqq_1h.empty or qqq_1d.empty:
        print("❌ 数据下载失败或数据不足！")
        return

    # 2. 计算 1 小时级别的 EMA30
    qqq_1h['EMA_30'] = qqq_1h['Close'].ewm(span=30, adjust=False).mean()
    tqqq_1h['EMA_30'] = tqqq_1h['Close'].ewm(span=30, adjust=False).mean()

    # 计算日线级别的 EMA50
    qqq_1d['Daily_EMA_50'] = qqq_1d['Close'].ewm(span=50, adjust=False).mean()

    # 🛠️ 核心量化技术：将日线指标映射对齐到 1小时 的数据表中
    # 提取日期进行匹配，并使用向前填充 (ffill) 确保盘中也能看到昨天的日线支撑位
    qqq_1h['Date_Only'] = qqq_1h.index.normalize()
    qqq_1d['Date_Only'] = qqq_1d.index.normalize()
    daily_ema_map = dict(zip(qqq_1d['Date_Only'], qqq_1d['Daily_EMA_50']))
    qqq_1h['Daily_EMA_50'] = qqq_1h['Date_Only'].map(daily_ema_map).ffill()

    # ================= 核心信号生成 =================

    # 🟢 信号 A+B：跨品种共振狠狠加仓
    # 条件：QQQ 刚好踩在它的 1h EMA30 上，同时 TQQQ 也精准回踩在它的 1h EMA30 上 (容错率 0.5%)
    qqq_hits_ema30 = (qqq_1h['Close'] >= qqq_1h['EMA_30'] * 0.995) & (qqq_1h['Close'] <= qqq_1h['EMA_30'] * 1.005)
    tqqq_hits_ema30 = (tqqq_1h['Close'] >= tqqq_1h['EMA_30'] * 0.995) & (tqqq_1h['Close'] <= tqqq_1h['EMA_30'] * 1.005)

    qqq_1h['Buy_Signal'] = np.where(qqq_hits_ema30 & tqqq_hits_ema30, 1, 0)

    # 🔴 信号 C：钻石手卖出
    # 条件：只有当 1小时 的价格彻底跌破了 日线级别 的 EMA50，才认输离场。过滤掉盘中假摔！
    qqq_1h['Sell_Signal'] = np.where(qqq_1h['Close'] < qqq_1h['Daily_EMA_50'], -1, 0)

    # ================= 状态与收益计算 =================
    qqq_1h['Signal'] = qqq_1h['Buy_Signal'].add(qqq_1h['Sell_Signal'], fill_value=0)

    # 修复旧版 pandas 的 FutureWarning
    qqq_1h['Position'] = qqq_1h['Signal'].replace(0, np.nan).ffill().fillna(0)
    qqq_1h['Position'] = qqq_1h['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 计算交易次数
    trades = qqq_1h['Position'].diff().abs().sum() / 2

    # 计算策略收益
    qqq_1h['Market_Return'] = qqq_1h['Close'].pct_change()
    qqq_1h['Strategy_Return'] = qqq_1h['Position'].shift(1) * qqq_1h['Market_Return']

    qqq_1h['Market_Equity'] = (1 + qqq_1h['Market_Return']).cumprod()
    qqq_1h['Strategy_Equity'] = (1 + qqq_1h['Strategy_Return']).cumprod()

    # ================= 输出回测报告 =================
    total_market_return = (qqq_1h['Market_Equity'].iloc[-1] - 1) * 100
    total_strategy_return = (qqq_1h['Strategy_Equity'].iloc[-1] - 1) * 100

    rolling_max = qqq_1h['Strategy_Equity'].cummax()
    max_drawdown = ((qqq_1h['Strategy_Equity'] - rolling_max) / rolling_max).min() * 100

    print("\n" + "="*50)
    print("💎 V7.0 钻石手 & 跨品种共振回测 (Educational Purpose)")
    print("="*50)
    print(f"标的: QQQ (1h Interval 结合 Daily EMA50 防守)")
    print(f"回测跨度: 过去 2 年 (730 days)")
    print(f"模拟交易次数: 约 {int(trades)} 次")
    print(f"策略总收益 (Strategy): {total_strategy_return:.2f}%")
    print(f"同期大盘收益 (Buy&Hold): {total_market_return:.2f}%")
    print(f"策略最大回撤 (Max Drawdown): {max_drawdown:.2f}%")
    print("="*50)

if __name__ == "__main__":
    run_v7_diamond_hands_backtest()

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层算法区 =================
def calculate_rsi(series, period):
    """纯手工编写的 RSI 算法"""
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    """计算 ATR (真实波动幅度)"""
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= 核心 A/B 测试回测引擎 =================
def run_v8_ab_test_backtest(ticker="QQQ", period="730d"):
    print(f"🚀 开始 V8.0 顺势捡漏模型 A/B 测试: {ticker} (1小时级别, 过去 {period})")

    # 1. 获取数据
    df = yf.Ticker(ticker).history(period=period, interval="1h")
    if df.empty or len(df) < 200:
        print("❌ 数据下载失败或数据不足！")
        return

    # 2. 计算基准均线 (牛市过滤网)
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_30'] = df['Close'].ewm(span=30, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

    # 🐂 牛市确认条件：多头排列 且 价格在EMA20之上
    bull_market_condition = (df['EMA_20'] > df['EMA_30']) & (df['EMA_30'] > df['EMA_50']) & (df['Close'] > df['EMA_20'])

    # ================= 方法1：短线情绪极度错杀 (Connors RSI) =================
    # 使用极其敏感的 3 小时 RSI
    df['RSI_3'] = calculate_rsi(df['Close'], period=3)
    # 买入：牛市背景下，3小时RSI跌破 20 (短线被错杀抛售)
    df['Buy_M1'] = np.where(bull_market_condition & (df['RSI_3'] < 20), 1, 0)

    # ================= 方法2：ATR 动态折扣法 (波动率通道) =================
    df['ATR_14'] = calculate_atr(df, period=14)
    df['Roll_High_20'] = df['High'].rolling(20).max() # 过去20小时的最高价
    # 买入：牛市背景下，价格从近期高点回撤了至少 1.5 倍的 ATR
    df['Buy_M2'] = np.where(bull_market_condition & (df['Close'] < (df['Roll_High_20'] - 1.5 * df['ATR_14'])), 1, 0)

    # ================= 坚决卖出 (止损/止盈) =================
    # 只要跌破 EMA20 生命周期，立刻清仓，绝不恋战！
    sell_signal = np.where(df['Close'] < df['EMA_20'], -1, 0)

    # ================= 评估函数 =================
    def evaluate_strategy(buy_column, method_name):
        # 合并信号
        sig = df[buy_column].add(pd.Series(sell_signal, index=df.index), fill_value=0)
        pos = sig.replace(0, np.nan).ffill().fillna(0)
        pos = pos.apply(lambda x: 1 if x > 0 else 0)

        # 计算交易次数
        trades = pos.diff().abs().sum() / 2

        # 计算收益
        market_return = df['Close'].pct_change()
        strat_return = pos.shift(1) * market_return

        strat_equity = (1 + strat_return).cumprod()
        market_equity = (1 + market_return).cumprod()

        total_strat_return = (strat_equity.iloc[-1] - 1) * 100
        total_market_return = (market_equity.iloc[-1] - 1) * 100

        rolling_max = strat_equity.cummax()
        max_drawdown = ((strat_equity - rolling_max) / rolling_max).min() * 100

        print("\n" + "-"*45)
        print(f"📊 {method_name}")
        print("-"*45)
        print(f"模拟交易次数: 约 {int(trades)} 次")
        print(f"策略总收益: {total_strat_return:.2f}% (大盘收益: {total_market_return:.2f}%)")
        print(f"策略最大回撤: {max_drawdown:.2f}%")
        return total_strat_return

    # 执行评估并打印报告
    evaluate_strategy('Buy_M1', "方法 1: RSI(3) 极度错杀流 (Connors Style)")
    evaluate_strategy('Buy_M2', "方法 2: ATR 动态折扣流 (Volatility Channel)")
    print("\n" + "="*45)

if __name__ == "__main__":
    # 跑跑 QQQ 看哪个方法更牛！
    run_v8_ab_test_backtest("QQQ", period="730d")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层算法区 =================
def calculate_rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= V9.0 中长线 A/B 测试引擎 =================
def run_v9_daily_swing_backtest(ticker="QQQ", period="5y"):
    print(f"🚀 开始 V9.0 日线中长线捡漏模型 A/B 测试: {ticker} (Daily 级别, 过去 {period})")
    print("⏳ 正在跨越牛熊周期 (包含2022年大熊市)...\n")

    # 1. 获取日线数据 (突破了小时线只能拉 730 天的限制！)
    df = yf.Ticker(ticker).history(period=period, interval="1d")
    if df.empty or len(df) < 200:
        print("❌ 数据下载失败或数据不足！")
        return

    # 2. 计算中长线基准均线
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_100'] = df['Close'].ewm(span=100, adjust=False).mean()

    # 🐂 大环境过滤：日线级别多头排列 (长期向上趋势)
    bull_market_condition = (df['EMA_20'] > df['EMA_50']) & (df['EMA_50'] > df['EMA_100']) & (df['Close'] > df['EMA_50'])

    # ================= 方法1：日线级别 RSI(3) 情绪恐慌 =================
    df['RSI_3'] = calculate_rsi(df['Close'], period=3)
    # 稍微放宽一点，日线RSI(3)低于 25 即视为有价值的砸坑
    df['Buy_M1'] = np.where(bull_market_condition & (df['RSI_3'] < 25), 1, 0)

    # ================= 方法2：日线 ATR 动态折扣流 =================
    df['ATR_14'] = calculate_atr(df, period=14)
    df['Roll_High_20'] = df['High'].rolling(20).max()
    # 价格从最近一个月的高点，回撤了至少 1.5 倍的日均波动幅度
    df['Buy_M2'] = np.where(bull_market_condition & (df['Close'] < (df['Roll_High_20'] - 1.5 * df['ATR_14'])), 1, 0)

    # ================= 坚决防守 (中长线防守) =================
    # 中长线不能频繁割肉！放宽到跌破日线 EMA50 (中线生命线) 才清仓防守！
    sell_signal = np.where(df['Close'] < df['EMA_50'], -1, 0)

    # ================= 评估函数 =================
    def evaluate_strategy(buy_column, method_name):
        sig = df[buy_column].add(pd.Series(sell_signal, index=df.index), fill_value=0)
        pos = sig.replace(0, np.nan).ffill().fillna(0)
        pos = pos.apply(lambda x: 1 if x > 0 else 0)

        trades = pos.diff().abs().sum() / 2

        market_return = df['Close'].pct_change()
        strat_return = pos.shift(1) * market_return

        strat_equity = (1 + strat_return).cumprod()
        market_equity = (1 + market_return).cumprod()

        total_strat_return = (strat_equity.iloc[-1] - 1) * 100
        total_market_return = (market_equity.iloc[-1] - 1) * 100

        rolling_max = strat_equity.cummax()
        max_drawdown = ((strat_equity - rolling_max) / rolling_max).min() * 100
        market_rolling_max = market_equity.cummax()
        market_max_drawdown = ((market_equity - market_rolling_max) / market_rolling_max).min() * 100

        print("-" * 55)
        print(f"📊 {method_name}")
        print("-" * 55)
        print(f"模拟交易次数: 约 {int(trades)} 次 (5年间，平均每年交易 {(int(trades)/5):.1f} 次)")
        print(f"策略总收益: {total_strat_return:.2f}% (无脑死扛收益: {total_market_return:.2f}%)")
        print(f"策略最大回撤: {max_drawdown:.2f}%  (无脑死扛最大回撤: {market_max_drawdown:.2f}%)")
        return total_strat_return

    evaluate_strategy('Buy_M1', "方法 1: 日线 RSI(3) 情绪错杀抄底")
    evaluate_strategy('Buy_M2', "方法 2: 日线 ATR 波动率折扣抄底")
    print("\n" + "="*55)

if __name__ == "__main__":
    run_v9_daily_swing_backtest("QQQ", period="5y")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层数学与指标库 =================
def calculate_rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= V12.0 机构级引擎 =================
def run_v12_quant_engine(ticker="QQQ", period="5y", slippage=0.001):
    print(f"🚀 开始 V12.0 机构级量化模型回测: {ticker} (Daily 级别, 过去 {period})")
    print(f"⚙️ 已开启: 动态波动率状态机 | 风险平价仓位管理 | T+1 滑点摩擦成本 ({slippage*100}%)\n")

    df = yf.Ticker(ticker).history(period=period, interval="1d")
    if df.empty or len(df) < 200:
        return

    # 1. 基础趋势线
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # 2. 核心数学模块：波动率状态机 (Regime Detection)
    df['ATR_14'] = calculate_atr(df, period=14)
    df['Norm_ATR'] = df['ATR_14'] / df['Close'] # 标准化波动率
    df['Long_Vol'] = df['Norm_ATR'].rolling(100).mean() # 过去100天平均波动率
    df['Vol_Ratio'] = df['Norm_ATR'] / df['Long_Vol'] # 波动率比值

    # 定义状态：Vol_Ratio > 1.2 为高危恐慌状态 (Regime: High Vol)

    # 3. 基础信号生成
    df['RSI_3'] = calculate_rsi(df['Close'], period=3)
    df['Roll_High_20'] = df['High'].rolling(20).max()

    # 买入条件：宏观牛市 (价格 > EMA200) 且出现超卖或打折
    bull_market = df['Close'] > df['EMA_200']
    buy_condition = bull_market & ((df['RSI_3'] < 30) | (df['Close'] < df['Roll_High_20'] - 1.5 * df['ATR_14']))

    # 卖出条件：跌破中期生命线
    sell_condition = df['Close'] < df['EMA_50']

    # 4. 核心数学模块：动态仓位控制 (Dynamic Sizing)
    # 如果触发买入，不是固定买 100%。如果处于高波动状态，只买入 50% 规避风险。
    df['Target_Position'] = np.where(buy_condition, np.where(df['Vol_Ratio'] > 1.2, 0.5, 1.0), np.nan)
    df['Target_Position'] = np.where(sell_condition, 0.0, df['Target_Position'])

    # 状态继承 (向前填充仓位)
    df['Position'] = df['Target_Position'].ffill().fillna(0.0)

    # 5. 核心数学模块：加入滑点与摩擦成本 (Slippage & Friction)
    # 每次仓位变化，都要扣除千分之一的滑点和手续费
    df['Trade_Friction'] = df['Position'].diff().abs() * slippage

    # 6. 计算最终真实收益
    df['Market_Return'] = df['Close'].pct_change()
    # 策略真实收益 = (昨天仓位 * 今天大盘涨跌) - 今天因调仓产生的摩擦成本
    df['Strategy_Return'] = (df['Position'].shift(1) * df['Market_Return']) - df['Trade_Friction'].fillna(0)

    df['Market_Equity'] = (1 + df['Market_Return']).cumprod()
    df['Strategy_Equity'] = (1 + df['Strategy_Return']).cumprod()

    # ================= 输出量化报告 =================
    total_strat_return = (df['Strategy_Equity'].iloc[-1] - 1) * 100
    total_market_return = (df['Market_Equity'].iloc[-1] - 1) * 100

    rolling_max = df['Strategy_Equity'].cummax()
    max_drawdown = ((df['Strategy_Equity'] - rolling_max) / rolling_max).min() * 100
    market_rolling_max = df['Market_Equity'].cummax()
    market_max_drawdown = ((df['Market_Equity'] - market_rolling_max) / market_rolling_max).min() * 100

    # 统计换手率 (Turnover)
    turnover = df['Position'].diff().abs().sum() / 2

    print("-" * 65)
    print("🔬 V12.0 机构级量化分析报告 (经摩擦成本调整后)")
    print("-" * 65)
    print(f"总换手次数 (Turnover): 约 {int(turnover)} 次")
    print(f"策略真实总收益: {total_strat_return:.2f}% (大盘基准: {total_market_return:.2f}%)")
    print(f"策略最大真实回撤: {max_drawdown:.2f}%  (大盘基准: {market_max_drawdown:.2f}%)")
    print("=" * 65)

if __name__ == "__main__":
    run_v12_quant_engine("QQQ", period="5y")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层算法区 =================
def calculate_rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= 杠铃策略网格搜索优化器 =================
def run_barbell_optimization(period="5y"):
    print("🚀 启动量化网格搜索：寻找最佳闲置资金 (TQQQ) 配比...\n")

    # 1. 获取标的数据
    qqq = yf.Ticker("QQQ").history(period=period, interval="1d")
    tqqq = yf.Ticker("TQQQ").history(period=period, interval="1d")

    # 数据对齐清洗
    df = pd.DataFrame({
        'QQQ_Close': qqq['Close'], 'QQQ_High': qqq['High'], 'QQQ_Low': qqq['Low'],
        'TQQQ_Close': tqqq['Close']
    }).dropna()

    # 2. V9.0 日线波段核心逻辑
    df['EMA_20'] = df['QQQ_Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['QQQ_Close'].ewm(span=50, adjust=False).mean()
    df['EMA_100'] = df['QQQ_Close'].ewm(span=100, adjust=False).mean()

    bull_market = (df['EMA_20'] > df['EMA_50']) & (df['EMA_50'] > df['EMA_100']) & (df['QQQ_Close'] > df['EMA_50'])

    df['RSI_3'] = calculate_rsi(df['QQQ_Close'], period=3)
    df['ATR_14'] = calculate_atr(df.rename(columns={'QQQ_High': 'High', 'QQQ_Low': 'Low', 'QQQ_Close': 'Close'}), period=14)
    df['Roll_High_20'] = df['QQQ_High'].rolling(20).max()

    buy_signal = bull_market & ((df['RSI_3'] < 25) | (df['QQQ_Close'] < (df['Roll_High_20'] - 1.5 * df['ATR_14'])))
    sell_signal = df['QQQ_Close'] < df['EMA_50']

    # 生成基础仓位 (在场内为1，在场外为0)
    df['Signal'] = np.where(buy_signal, 1, np.where(sell_signal, -1, 0))
    df['Position'] = df['Signal'].replace(0, np.nan).ffill().fillna(0)
    df['Position'] = df['Position'].apply(lambda x: 1 if x > 0 else 0)

    # 3. 收益率计算准备
    df['QQQ_Ret'] = df['QQQ_Close'].pct_change()
    df['TQQQ_Ret'] = df['TQQQ_Close'].pct_change()
    daily_rf = 0.045 / 252 # 模拟 4.5% 的无风险国债日化收益

    # 4. 网格搜索测试配比
    print(f"{'空仓期TQQQ配比':<14} | {'总收益率':<10} | {'最大回撤':<10} | {'夏普比率 (风控)':<10}")
    print("-" * 65)

    # 我们测试从 0% 一直到 30% 的 TQQQ 配比
    test_ratios = [0.0, 0.05, 0.08, 0.10, 0.11, 0.12, 0.15, 0.20, 0.30]

    for ratio in test_ratios:
        # 在场内时：100% 享受 QQQ 的涨跌
        active_ret = df['Position'].shift(1) * df['QQQ_Ret']
        # 在场外闲置时：ratio 承受 TQQQ 涨跌 + (1-ratio) 享受国债每天稳稳的利息
        idle_ret = (1 - df['Position'].shift(1)) * (ratio * df['TQQQ_Ret'] + (1 - ratio) * daily_rf)

        # 每天的综合真实收益
        strat_ret = active_ret + idle_ret

        # 净值计算
        equity = (1 + strat_ret).cumprod()
        total_return = (equity.iloc[-1] - 1) * 100

        # 回撤与夏普
        rolling_max = equity.cummax()
        drawdown = ((equity - rolling_max) / rolling_max).min() * 100

        annualized_ret = strat_ret.mean() * 252
        annualized_vol = strat_ret.std() * np.sqrt(252)
        sharpe = (annualized_ret - 0.045) / annualized_vol if annualized_vol != 0 else 0

        print(f" {int(ratio*100):>2}% TQQQ       | {total_return:>7.2f}%   | {drawdown:>8.2f}%  |  {sharpe:>5.2f}")

    print("-" * 65)
    # 大盘基准对照
    market_equity = (1 + df['QQQ_Ret']).cumprod()
    mkt_ret = (market_equity.iloc[-1] - 1) * 100
    mkt_dd = ((market_equity - market_equity.cummax()) / market_equity.cummax()).min() * 100
    print(f" 大盘 (死扛QQQ)   | {mkt_ret:>7.2f}%   | {mkt_dd:>8.2f}%  |   N/A\n")

if __name__ == "__main__":
    run_barbell_optimization()

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层算法区 =================
def calculate_rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= V10.0 利润提速版引擎 =================
def run_v10_profit_boost_backtest(ticker="QQQ", period="5y"):
    print(f"🚀 开始 V10.0 利润提速模型 A/B 测试: {ticker} (Daily 级别, 过去 {period})")

    df = yf.Ticker(ticker).history(period=period, interval="1d")
    if df.empty or len(df) < 200:
        return

    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

    # 🏎️ 优化 1：放宽大环境过滤，不再死等三线发散，上车更快
    bull_market_condition = (df['EMA_20'] > df['EMA_50']) & (df['Close'] > df['EMA_50'])

    # ================= 方法1：RSI(3) 门槛降低 =================
    df['RSI_3'] = calculate_rsi(df['Close'], period=3)
    # 🏎️ 优化 2：RSI(3) 从 < 25 放宽到 < 35，捕捉更多浅回调
    df['Buy_M1'] = np.where(bull_market_condition & (df['RSI_3'] < 35), 1, 0)

    # ================= 方法2：ATR 折扣门槛降低 =================
    df['ATR_14'] = calculate_atr(df, period=14)
    df['Roll_High_20'] = df['High'].rolling(20).max()
    # 🏎️ 优化 3：回撤从 1.5 倍 ATR 降低到 1.0 倍 ATR
    df['Buy_M2'] = np.where(bull_market_condition & (df['Close'] < (df['Roll_High_20'] - 1.0 * df['ATR_14'])), 1, 0)

    # ================= 坚决防守 (中线生命线) =================
    sell_signal = np.where(df['Close'] < df['EMA_50'], -1, 0)

    def evaluate_strategy(buy_column, method_name):
        sig = df[buy_column].add(pd.Series(sell_signal, index=df.index), fill_value=0)
        pos = sig.replace(0, np.nan).ffill().fillna(0)
        pos = pos.apply(lambda x: 1 if x > 0 else 0)

        trades = pos.diff().abs().sum() / 2

        market_return = df['Close'].pct_change()
        strat_return = pos.shift(1) * market_return

        strat_equity = (1 + strat_return).cumprod()
        market_equity = (1 + market_return).cumprod()

        total_strat_return = (strat_equity.iloc[-1] - 1) * 100
        total_market_return = (market_equity.iloc[-1] - 1) * 100

        rolling_max = strat_equity.cummax()
        max_drawdown = ((strat_equity - rolling_max) / rolling_max).min() * 100
        market_rolling_max = market_equity.cummax()
        market_max_drawdown = ((market_equity - market_rolling_max) / market_rolling_max).min() * 100

        print("-" * 55)
        print(f"📈 {method_name}")
        print("-" * 55)
        print(f"模拟交易次数: 约 {int(trades)} 次")
        print(f"策略总收益: {total_strat_return:.2f}% (大盘收益: {total_market_return:.2f}%)")
        print(f"策略最大回撤: {max_drawdown:.2f}%  (大盘回撤: {market_max_drawdown:.2f}%)")
        return total_strat_return

    evaluate_strategy('Buy_M1', "方法 1: 日线 RSI(3) < 35 浅回调抄底")
    evaluate_strategy('Buy_M2', "方法 2: 日线 1.0x ATR 正常波动抄底")
    print("\n" + "="*55)

if __name__ == "__main__":
    run_v10_profit_boost_backtest("QQQ", period="5y")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ================= 底层算法区 =================
def calculate_rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(com=period - 1, adjust=False).mean()
    roll_down = down.ewm(com=period - 1, adjust=False).mean()
    rs = roll_up / roll_down
    return 100 - (100 / (1 + rs))

def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ================= V11.0 反向调整：狙击手引擎 =================
def run_v11_sniper_backtest(ticker="QQQ", period="5y"):
    print(f"🚀 开始 V11.0 反向调整(狙击手模型) A/B 测试: {ticker} (Daily 级别, 过去 {period})")

    df = yf.Ticker(ticker).history(period=period, interval="1d")
    if df.empty or len(df) < 200:
        return

    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_100'] = df['Close'].ewm(span=100, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean() # 引入终极牛市过滤线

    # 🦅 反向调整 1：宏观过滤极其严格，必须在 EMA200 之上，且短期均线向上
    bull_market_condition = (df['Close'] > df['EMA_200']) & (df['EMA_20'] > df['EMA_50'])

    # ================= 方法1：RSI(3) 极限恐慌 =================
    df['RSI_3'] = calculate_rsi(df['Close'], period=3)
    # 🦅 反向调整 2：RSI(3) 从 < 35 重新收紧到 < 20，只抓带血的筹码
    df['Buy_M1'] = np.where(bull_market_condition & (df['RSI_3'] < 20), 1, 0)

    # ================= 方法2：ATR 闪崩折扣 =================
    df['ATR_14'] = calculate_atr(df, period=14)
    df['Roll_High_20'] = df['High'].rolling(20).max()
    # 🦅 反向调整 3：回撤要求从 1.0 倍提高到 2.0 倍 ATR（深坑抄底）
    df['Buy_M2'] = np.where(bull_market_condition & (df['Close'] < (df['Roll_High_20'] - 2.0 * df['ATR_14'])), 1, 0)

    # ================= 宽容防守 (让利润狂奔) =================
    # 🦅 反向调整 4：防守线从 EMA50 拓宽到 EMA100！过滤掉中级洗盘，死拿主升浪！
    sell_signal = np.where(df['Close'] < df['EMA_100'], -1, 0)

    def evaluate_strategy(buy_column, method_name):
        sig = df[buy_column].add(pd.Series(sell_signal, index=df.index), fill_value=0)
        pos = sig.replace(0, np.nan).ffill().fillna(0)
        pos = pos.apply(lambda x: 1 if x > 0 else 0)

        trades = pos.diff().abs().sum() / 2

        market_return = df['Close'].pct_change()
        strat_return = pos.shift(1) * market_return

        strat_equity = (1 + strat_return).cumprod()
        market_equity = (1 + market_return).cumprod()

        total_strat_return = (strat_equity.iloc[-1] - 1) * 100
        total_market_return = (market_equity.iloc[-1] - 1) * 100

        rolling_max = strat_equity.cummax()
        max_drawdown = ((strat_equity - rolling_max) / rolling_max).min() * 100
        market_rolling_max = market_equity.cummax()
        market_max_drawdown = ((market_equity - market_rolling_max) / market_rolling_max).min() * 100

        print("-" * 55)
        print(f"🎯 {method_name}")
        print("-" * 55)
        print(f"模拟交易次数: 约 {int(trades)} 次")
        print(f"策略总收益: {total_strat_return:.2f}% (大盘收益: {total_market_return:.2f}%)")
        print(f"策略最大回撤: {max_drawdown:.2f}%  (大盘回撤: {market_max_drawdown:.2f}%)")
        return total_strat_return

    evaluate_strategy('Buy_M1', "方法 1: 日线 RSI(3) < 20 极限带血抄底")
    evaluate_strategy('Buy_M2', "方法 2: 日线 2.0x ATR 闪崩深坑抄底")
    print("\n" + "="*55)

if __name__ == "__main__":
    run_v11_sniper_backtest("QQQ", period="5y")